In [ ]:
RESNET_CONFIDENCE = 0.7
EXECUTE_PIPELINE = True

print(f"Mode: {'ACTIVE' if EXECUTE_PIPELINE else 'DRY RUN'}")
print(f"ResNet confidence threshold: {RESNET_CONFIDENCE}")


In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import json
import shutil

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "detected_charts"

OUTPUT_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "classified_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")


In [ ]:
input_images = list(INPUT_DIR.glob("*.png")) + list(INPUT_DIR.glob("*.jpg"))

print("=" * 50)
print("INPUT STATUS")
print("=" * 50)
print(f"Charts to classify: {len(input_images):,}")
print("=" * 50)

if len(input_images) == 0:
    print("\n[WARNING] No images found!")
    print(f"Run 01c_chart_detection.ipynb first.")


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

resnet_path = PROJECT_ROOT / "models" / "weights" / "resnet18_chart_classifier_best.pt"

if not resnet_path.exists():
    raise FileNotFoundError(f"ResNet model not found: {resnet_path}")

checkpoint = torch.load(resnet_path, map_location=device, weights_only=False)

class_mapping = checkpoint.get("class_mapping", {})
CHART_TYPES = [k for k, v in sorted(class_mapping.items(), key=lambda x: x[1])]
print(f"Chart types: {CHART_TYPES}")

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CHART_TYPES))

state_dict = checkpoint.get("model_state_dict", checkpoint)
if any(k.startswith("resnet.") for k in state_dict.keys()):
    state_dict = {k.replace("resnet.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.to(device).eval()

print(f"ResNet-18 loaded | val_acc={checkpoint.get('val_acc', 'N/A')}%")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
import random
import matplotlib.pyplot as plt


def classify_chart(image_path: Path):
    """Classify single chart image."""
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)[0]
        conf, pred_idx = probs.max(0)
    
    return CHART_TYPES[pred_idx.item()], conf.item()


if input_images:
    samples = random.sample(input_images, min(8, len(input_images)))
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for ax, img_path in zip(axes, samples):
        chart_type, conf = classify_chart(img_path)
        
        img = Image.open(img_path)
        ax.imshow(img)
        
        color = "green" if conf >= RESNET_CONFIDENCE else "orange"
        ax.set_title(f"{chart_type}\n({conf:.0%})", color=color, fontsize=11, fontweight="bold")
        ax.axis("off")
    
    plt.suptitle("ResNet-18 Classification Samples", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
from tqdm.auto import tqdm
import time
from typing import Dict


def classify_and_organize(
    input_dir: Path,
    output_dir: Path,
    min_confidence: float = 0.7,
    dry_run: bool = True,
) -> Dict:
    """
    Classify charts and organize by type.
    """
    images = list(input_dir.glob("*.png")) + list(input_dir.glob("*.jpg"))
    
    print("=" * 60)
    print(f"CHART CLASSIFICATION | {'DRY RUN' if dry_run else 'ACTIVE'}")
    print(f"Images: {len(images):,}")
    print(f"Min confidence: {min_confidence:.0%}")
    print("=" * 60)
    
    if dry_run:
        print("\n[DRY RUN] Set EXECUTE_PIPELINE = True to run.")
        return {}
    
    for ct in CHART_TYPES + ["uncertain"]:
        (output_dir / ct).mkdir(parents=True, exist_ok=True)
    
    results = []
    by_type = {ct: 0 for ct in CHART_TYPES + ["uncertain"]}
    
    start = time.time()
    
    for img_path in tqdm(images, desc="Classifying"):
        try:
            chart_type, confidence = classify_chart(img_path)
            
            dest_type = chart_type if confidence >= min_confidence else "uncertain"
            
            dest_path = output_dir / dest_type / img_path.name
            shutil.copy2(img_path, dest_path)
            
            by_type[dest_type] += 1
            
            results.append({
                "image": img_path.name,
                "type": chart_type,
                "confidence": round(confidence, 4),
                "dest": dest_type,
            })
            
        except Exception as e:
            print(f"Error: {img_path.name} - {e}")
    
    elapsed = time.time() - start
    
    results_path = output_dir / "classification_results.json"
    with open(results_path, "w") as f:
        json.dump({"results": results, "by_type": by_type}, f, indent=2)
    
    print("\n" + "=" * 60)
    print("CLASSIFICATION COMPLETE")
    print(f"Time: {elapsed:.1f}s ({len(images)/elapsed:.1f} img/s)")
    print("-" * 60)
    print("Distribution:")
    for ct, count in sorted(by_type.items(), key=lambda x: -x[1]):
        if count > 0:
            pct = count / len(images) * 100
            print(f"  {ct:12s}: {count:5,} ({pct:5.1f}%)")
    print("=" * 60)
    
    return {"results": results, "by_type": by_type}


In [ ]:
if EXECUTE_PIPELINE and len(input_images) > 0:
    classification_results = classify_and_organize(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        min_confidence=RESNET_CONFIDENCE,
        dry_run=False,
    )
else:
    classification_results = {}
    print("[SKIPPED]")


In [ ]:
import matplotlib.pyplot as plt

if classification_results:
    by_type = classification_results["by_type"]
    
    types = [t for t, c in by_type.items() if c > 0]
    counts = [by_type[t] for t in types]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(types, counts, color="steelblue")
    ax.set_xlabel("Count")
    ax.set_title("Chart Type Distribution")
    
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f"{count:,}", va="center", fontsize=10)
    
    plt.tight_layout()
    plt.show()


In [ ]:
print("=" * 60)
print("CLASSIFICATION SUMMARY")
print("=" * 60)
print(f"\nOutput: {OUTPUT_DIR}/")

total = 0
for ct in CHART_TYPES + ["uncertain"]:
    ct_dir = OUTPUT_DIR / ct
    if ct_dir.exists():
        count = len(list(ct_dir.glob("*.png"))) + len(list(ct_dir.glob("*.jpg")))
        if count > 0:
            print(f"  {ct}/  ({count:,})")
            total += count

print(f"\nTotal: {total:,} charts classified")
print("=" * 60)